# Complete Bone Fracture Weakly Supervised Segmentation Pipeline

**One notebook** covering:
1. Load fracture/non-fracture X-ray classification data.
2. Train/evaluate one of 15 image-classification backbones.
3. Generate Grad-CAM heatmaps and binary pseudo-masks.
4. Train U-Net and DeepLabV3+ using training pseudo-masks.
5. Optionally evaluate segmentation against a separate expert-annotated test set.

> Important: This is a starter research pipeline. Do patient-level splitting if patient IDs are available. Do not generate training pseudo-masks from held-out test images. Keep expert test masks separate until final evaluation. Grad-CAM is only implemented for the supported CNN backbones listed below; the transformer/CLIP/RadDINO models can classify images, but need architecture-specific explainability methods for reliable CAM.


## 1. Install and import

In [ ]:
!pip -q install timm segmentation-models-pytorch opencv-python-headless scikit-learn pandas matplotlib pillow


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Configure paths and settings

In [ ]:
import os, random, copy, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Expected classification folders:
# CLASSIFICATION_DIR/
#   fractured/xxx.png
#   not_fractured/yyy.png
CLASSIFICATION_DIR = "/content/drive/MyDrive/bone_fracture/classification_data"

# Training X-rays and their generated pseudo masks for segmentation:
TRAIN_IMAGES_DIR = "/content/drive/MyDrive/bone_fracture/train_images"
PSEUDO_MASKS_DIR = "/content/drive/MyDrive/bone_fracture/pseudo_masks"

# Optional held-out expert-labeled test data. Leave None until ready.
TEST_IMAGES_DIR = None
EXPERT_MASKS_DIR = None

OUTPUT_DIR = "/content/drive/MyDrive/bone_fracture/complete_pipeline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
SEGMENTATION_SIZE = 256
BATCH_SIZE = 16
CLASSIFIER_EPOCHS = 5
SEGMENTATION_EPOCHS = 30
LEARNING_RATE = 1e-4
NUM_WORKERS = 2

print("Device:", DEVICE)


## 4. Classification dataset and loaders

In [ ]:
class_names = ["fractured", "not_fractured"]
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

if not Path(CLASSIFICATION_DIR).exists():
    raise FileNotFoundError(f"Update CLASSIFICATION_DIR first: {CLASSIFICATION_DIR}")

# Use separate dataset objects so validation does not receive random augmentation.
train_full = datasets.ImageFolder(CLASSIFICATION_DIR, transform=train_transform)
eval_full = datasets.ImageFolder(CLASSIFICATION_DIR, transform=eval_transform)
print("Class mapping:", train_full.class_to_idx)
assert len(train_full.classes) == 2, "Expected exactly two class folders."

n = len(train_full)
if n < 2:
    raise ValueError("Need at least two images for a train/validation split.")
n_val = max(1, int(0.2*n))
n_train = n - n_val
indices = torch.randperm(n, generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices, val_indices = indices[:n_train], indices[n_train:]
from torch.utils.data import Subset
train_ds = Subset(train_full, train_indices)
val_ds = Subset(eval_full, val_indices)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE.type=="cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(DEVICE.type=="cuda"))
print(f"Train: {len(train_ds)} | Validation: {len(val_ds)}")


## 5. Build all 15 classifiers

The factory uses compatible torchvision/timm models. CLIP and RadDINO are optional and are only included if their dependencies/model weights can be loaded. You can select one model with `MODEL_NAME`; training all 15 sequentially can require substantial GPU time.

In [ ]:
NUM_CLASSES = 2

MODEL_NAMES = [
    "ConvNeXt-Tiny", "DenseNet121", "DenseNet201", "ResNet50", "ResNet101",
    "EfficientNet-B0", "EfficientNet-B3", "EfficientNet-B7", "MobileNetV2",
    "ViT-B-16", "ViT-B-32", "DeiT-Small", "Swin-Tiny", "CLIP-ViT-B-32", "RadDINO"
]

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def replace_head(model, name, nclasses=2):
    # Torchvision architectures
    if name.startswith("ResNet"):
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_features, nclasses))
    elif name.startswith("DenseNet"):
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.25), nn.Linear(in_features, nclasses))
    elif name.startswith("EfficientNet"):
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, nclasses)
    elif name == "MobileNetV2":
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, nclasses)
    return model

def build_classifier(name, nclasses=2):
    if name == "ConvNeXt-Tiny":
        m = timm.create_model("convnext_tiny", pretrained=True, num_classes=nclasses)
    elif name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m = replace_head(m, name, nclasses)
    elif name == "DenseNet201":
        m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        m = replace_head(m, name, nclasses)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m = replace_head(m, name, nclasses)
    elif name == "ResNet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        m = replace_head(m, name, nclasses)
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m = replace_head(m, "EfficientNetB0", nclasses)
    elif name == "EfficientNet-B3":
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        m = replace_head(m, "EfficientNetB3", nclasses)
    elif name == "EfficientNet-B7":
        m = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        m = replace_head(m, "EfficientNetB7", nclasses)
    elif name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m = replace_head(m, name, nclasses)
    elif name == "ViT-B-16":
        m = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        m.heads.head = nn.Linear(m.heads.head.in_features, nclasses)
    elif name == "ViT-B-32":
        m = timm.create_model("vit_base_patch32_224", pretrained=True, num_classes=nclasses)
    elif name == "DeiT-Small":
        m = timm.create_model("deit_small_patch16_224", pretrained=True, num_classes=nclasses)
    elif name == "Swin-Tiny":
        m = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=nclasses)
    elif name == "CLIP-ViT-B-32":
        # OpenAI CLIP is optional. If unavailable, install open_clip_torch and use a custom adapter.
        try:
            import clip
        except ImportError as e:
            raise ImportError("CLIP selected but package is missing. Run: !pip install git+https://github.com/openai/CLIP.git") from e
        clip_model, _ = clip.load("ViT-B/32", device="cpu", jit=False)
        freeze_all(clip_model)
        class ClipClassifier(nn.Module):
            def __init__(self, clip_model):
                super().__init__()
                self.clip_model = clip_model
                self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, nclasses))
            def forward(self, x):
                with torch.no_grad():
                    z = self.clip_model.encode_image(x).float()
                return self.classifier(z)
        m = ClipClassifier(clip_model)
        # This adapter's visual normalization should ideally use CLIP's own preprocessing.
    elif name == "RadDINO":
        m = timm.create_model("hf_hub:Snarcy/RadioDino-s16", pretrained=True, num_classes=nclasses)
    else:
        raise ValueError(f"Unknown model: {name}")

    # Freeze backbone and train classification head only, where head params are identifiable.
    if name not in ("CLIP-ViT-B-32",):
        freeze_all(m)
        if hasattr(m, "head") and isinstance(m.head, nn.Module):
            for p in m.head.parameters(): p.requires_grad = True
        if hasattr(m, "head_dist") and isinstance(m.head_dist, nn.Module):
            for p in m.head_dist.parameters(): p.requires_grad = True
        if hasattr(m, "heads"):
            for p in m.heads.parameters(): p.requires_grad = True
        if hasattr(m, "fc"):
            for p in m.fc.parameters(): p.requires_grad = True
        if hasattr(m, "classifier"):
            for p in m.classifier.parameters(): p.requires_grad = True
        # timm classifier access
        if hasattr(m, "get_classifier"):
            try:
                for p in m.get_classifier().parameters(): p.requires_grad = True
            except Exception:
                pass
    return m.to(DEVICE)

print("Available model choices:")
for x in MODEL_NAMES: print(" -", x)


## 6. Train and validate a selected classifier

In [ ]:
def train_classifier(model, train_loader, val_loader, epochs=CLASSIFIER_EPOCHS, save_path=None):
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    if not params:
        raise RuntimeError("No trainable parameters. Check classifier head freezing.")
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=1e-4)
    best_acc = -1
    history = []
    for epoch in range(1, epochs+1):
        model.train()
        train_loss, train_correct, train_n = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()*images.size(0)
            train_correct += (logits.argmax(1)==labels).sum().item()
            train_n += labels.size(0)

        model.eval()
        val_loss, all_y, all_pred = 0.0, [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                logits = model(images)
                val_loss += criterion(logits, labels).item()*images.size(0)
                all_y.extend(labels.cpu().numpy().tolist())
                all_pred.extend(logits.argmax(1).cpu().numpy().tolist())
        val_acc = accuracy_score(all_y, all_pred)
        history.append({"epoch": epoch, "train_loss": train_loss/max(train_n,1),
                        "train_acc": train_correct/max(train_n,1),
                        "val_loss": val_loss/max(len(all_y),1), "val_acc": val_acc})
        print(f"Epoch {epoch}/{epochs} | train loss {history[-1]['train_loss']:.4f} "
              f"acc {history[-1]['train_acc']:.4f} | val loss {history[-1]['val_loss']:.4f} acc {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            if save_path:
                torch.save({"model_state_dict": model.state_dict(), "model_name": MODEL_NAME,
                            "class_to_idx": train_full.class_to_idx, "best_val_acc": best_acc}, save_path)
    return pd.DataFrame(history)

# Choose one:
MODEL_NAME = "ResNet50"  # choose from MODEL_NAMES
classifier = build_classifier(MODEL_NAME)
classifier_path = os.path.join(OUTPUT_DIR, f"{MODEL_NAME.replace('/', '-')}_classifier_best.pth")
classifier_history = train_classifier(classifier, train_loader, val_loader, save_path=classifier_path)
classifier_history


## 7. Classification validation report

In [ ]:
def classification_report_for_model(model, loader):
    model.eval()
    ys, preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            ys.extend(labels.numpy().tolist())
            preds.extend(logits.argmax(1).cpu().numpy().tolist())
    print(classification_report(ys, preds, target_names=[k for k,v in sorted(train_full.class_to_idx.items(), key=lambda x:x[1])], zero_division=0))
    cm = confusion_matrix(ys, preds)
    plt.figure(figsize=(5,4))
    plt.imshow(cm, cmap="Blues")
    plt.title("Validation confusion matrix")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(range(len(train_full.classes)), train_full.classes, rotation=30)
    plt.yticks(range(len(train_full.classes)), train_full.classes)
    for (i,j), val in np.ndenumerate(cm): plt.text(j, i, str(val), ha="center", va="center")
    plt.tight_layout(); plt.show()

classification_report_for_model(classifier, val_loader)


## 8. Grad-CAM for supported CNNs

This Grad-CAM implementation works for ResNet, DenseNet, and MobileNetV2. For ConvNeXt/EfficientNet, change `target_layer` to the final feature block as shown in the mapping. Transformer, CLIP, and RadDINO need architecture-specific CAM methods and are not supported by this cell.

In [ ]:
def get_gradcam_layer(model, model_name):
    if model_name.startswith("ResNet"):
        return model.layer4[-1]
    if model_name.startswith("DenseNet"):
        return model.features.denseblock4
    if model_name == "MobileNetV2":
        return model.features[-1]
    if model_name == "ConvNeXt-Tiny":
        return model.stages[-1]
    if model_name.startswith("EfficientNet"):
        return model.features[-1]
    raise ValueError(f"Grad-CAM target layer not configured for {model_name}")

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.hook1 = target_layer.register_forward_hook(self._save_activation)
        self.hook2 = target_layer.register_full_backward_hook(self._save_gradient)
    def _save_activation(self, module, inp, out):
        self.activations = out
    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]
    def __call__(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None: class_idx = int(logits.argmax(1).item())
        logits[:, class_idx].sum().backward()
        acts, grads = self.activations, self.gradients
        weights = grads.mean(dim=(2,3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True).relu()
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0,0].detach().cpu().numpy()
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, logits.detach()

def show_gradcam_for_dataset(model, model_name, dataset, idx=0, threshold=0.35):
    layer = get_gradcam_layer(model, model_name)
    cam_engine = GradCAM(model, layer)
    # Dataset item is image, label; Subset returns underlying transformed image.
    image, label = dataset[idx]
    x = image.unsqueeze(0).to(DEVICE)
    model.eval()
    cam, logits = cam_engine(x)
    heat = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)/255.0
    gray = image.permute(1,2,0).numpy().mean(axis=2)
    overlay = np.clip(0.55*gray[...,None] + 0.45*heat, 0, 1)
    pseudo = (cam >= threshold).astype(np.uint8)*255
    plt.figure(figsize=(14,4))
    for i,(im,title) in enumerate([(gray,"X-ray"),(cam,"Grad-CAM"),(overlay,"Overlay"),(pseudo,"Pseudo-mask")],1):
        plt.subplot(1,4,i); plt.imshow(im, cmap="gray" if i in (1,2,4) else None); plt.title(title); plt.axis("off")
    plt.show()
    return cam, pseudo

# Use validation image for a quick visual check:
# cam, pseudo_mask = show_gradcam_for_dataset(classifier, MODEL_NAME, val_ds, idx=0)


## 9. Generate pseudo-masks from training images

Use the **trained classifier** and only the training-image folder here. This function uses Grad-CAM for supported CNNs, resizes the heatmap to each source image, and thresholds it. This simple threshold is a baseline; tune it using training/validation data only.

In [ ]:
def generate_pseudo_masks(model, model_name, image_dir, output_mask_dir, threshold=0.35):
    os.makedirs(output_mask_dir, exist_ok=True)
    layer = get_gradcam_layer(model, model_name)
    cam_engine = GradCAM(model, layer)
    paths = [p for p in Path(image_dir).rglob("*") if p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".tif",".tiff"}]
    model.eval()
    count = 0
    for path in paths:
        raw = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if raw is None: continue
        resized = cv2.resize(raw, (IMAGE_SIZE, IMAGE_SIZE)).astype(np.float32)/255.0
        arr = np.stack([resized,resized,resized], axis=0)
        x = torch.from_numpy(arr).float().unsqueeze(0).to(DEVICE)
        # Only create fracture heatmaps for images predicted as class index 0.
        with torch.no_grad():
            pred_class = int(model(x).argmax(1).item())
        if pred_class != train_full.class_to_idx.get("fractured", 0):
            # save an all-background mask for predicted non-fracture cases
            mask = np.zeros(raw.shape, dtype=np.uint8)
        else:
            cam, _ = cam_engine(x, class_idx=pred_class)
            mask = (cv2.resize(cam, (raw.shape[1], raw.shape[0]) ) >= threshold).astype(np.uint8)*255
        out_path = Path(output_mask_dir) / f"{path.stem}.png"
        cv2.imwrite(str(out_path), mask)
        count += 1
    print(f"Saved {count} pseudo-masks to {output_mask_dir}")

# IMPORTANT: configure a TRAIN-ONLY image folder before running this.
# generate_pseudo_masks(classifier, MODEL_NAME, TRAIN_IMAGES_DIR, PSEUDO_MASKS_DIR, threshold=0.35)


## 10. Segmentation dataset and loaders

The segmentation images and pseudo masks must have matching filename stems. This split is a simple random image split; for clinical research, use a patient-level split.

In [ ]:
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def file_index(folder):
    folder = Path(folder)
    if not folder.exists(): raise FileNotFoundError(f"Folder not found: {folder}")
    d={}
    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS: d.setdefault(p.stem,p)
    return d

class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=256, augment=False):
        imgs, masks = file_index(image_dir), file_index(mask_dir)
        self.pairs = [(p, masks[s]) for s,p in sorted(imgs.items()) if s in masks]
        if not self.pairs: raise RuntimeError("No paired image/mask stems found. Check folders and filenames.")
        self.size, self.augment = size, augment
        print(f"Segmentation pairs: {len(self.pairs)} from {image_dir}")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ip, mp = self.pairs[i]
        image=cv2.imread(str(ip),cv2.IMREAD_GRAYSCALE); mask=cv2.imread(str(mp),cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None: raise RuntimeError(f"Read error for {ip} or {mp}")
        image=cv2.resize(image,(self.size,self.size),interpolation=cv2.INTER_AREA).astype(np.float32)/255
        mask=cv2.resize(mask,(self.size,self.size),interpolation=cv2.INTER_NEAREST)
        mask=(mask>0).astype(np.float32)
        if self.augment:
            if random.random()<0.5: image=image[:,::-1].copy(); mask=mask[:,::-1].copy()
            if random.random()<0.3: image=np.clip(image*random.uniform(.9,1.1),0,1)
        image=np.stack([image,image,image],axis=0)
        return torch.from_numpy(image.copy()).float(), torch.from_numpy(mask[None].copy()).float(), str(ip)

seg_full = SegmentationDataset(TRAIN_IMAGES_DIR, PSEUDO_MASKS_DIR, size=SEGMENTATION_SIZE, augment=False)
if len(seg_full)<2: raise ValueError("Need at least 2 image-mask pairs for segmentation train/validation.")
n_val=max(1,int(.2*len(seg_full))); n_train=len(seg_full)-n_val
tr_idx, va_idx = random_split(range(len(seg_full)), [n_train,n_val], generator=torch.Generator().manual_seed(SEED))
train_seg_ds=copy.copy(seg_full); train_seg_ds.augment=True
val_seg_ds=copy.copy(seg_full); val_seg_ds.augment=False
# random_split(range(...)) returns subsets whose indices reference range; use the index lists directly
train_seg_indices=list(tr_idx); val_seg_indices=list(va_idx)
train_seg_loader=DataLoader(Subset(train_seg_ds,train_seg_indices),batch_size=8,shuffle=True,num_workers=NUM_WORKERS)
val_seg_loader=DataLoader(Subset(val_seg_ds,val_seg_indices),batch_size=8,shuffle=False,num_workers=NUM_WORKERS)
print("Seg train:",len(train_seg_indices),"Seg validation:",len(val_seg_indices))


## 11. U-Net / DeepLabV3+ training

In [ ]:
import segmentation_models_pytorch as smp

class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth=smooth; self.bce=nn.BCEWithLogitsLoss()
    def forward(self, logits, targets):
        bce=self.bce(logits,targets); probs=torch.sigmoid(logits)
        dims=(1,2,3); inter=(probs*targets).sum(dims)
        dice=(2*inter+self.smooth)/(probs.sum(dims)+targets.sum(dims)+self.smooth)
        return bce+(1-dice).mean()

def seg_dice_iou(logits, targets, threshold=.5, eps=1e-7):
    pred=(torch.sigmoid(logits)>=threshold).float(); dims=(1,2,3)
    inter=(pred*targets).sum(dims)
    dice=(2*inter+eps)/(pred.sum(dims)+targets.sum(dims)+eps)
    iou=(inter+eps)/(pred.sum(dims)+targets.sum(dims)-inter+eps)
    return dice.mean().item(),iou.mean().item()

def build_segmentation_model(name):
    if name=="UNet":
        net=smp.Unet(encoder_name="resnet34",encoder_weights="imagenet",in_channels=3,classes=1,activation=None)
    elif name=="DeepLabV3Plus":
        net=smp.DeepLabV3Plus(encoder_name="resnet34",encoder_weights="imagenet",in_channels=3,classes=1,activation=None)
    else: raise ValueError("Use UNet or DeepLabV3Plus")
    return net.to(DEVICE)

def seg_epoch(model,loader,optimizer=None):
    training=optimizer is not None; model.train(training); criterion=DiceBCELoss()
    sums=np.zeros(3,dtype=float); count=0
    for x,y,_ in loader:
        x,y=x.to(DEVICE),y.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(x); loss=criterion(logits,y)
            if training: loss.backward(); optimizer.step()
        dice,iou=seg_dice_iou(logits.detach(),y)
        b=x.size(0); sums += np.array([loss.item(),dice,iou])*b; count+=b
    return sums/max(count,1)

def train_segmentation(name, epochs=SEGMENTATION_EPOCHS):
    model=build_segmentation_model(name)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    best=-1; history=[]; path=os.path.join(OUTPUT_DIR,f"{name}_best.pth")
    for epoch in range(1,epochs+1):
        tr=seg_epoch(model,train_seg_loader,optimizer)
        va=seg_epoch(model,val_seg_loader,None)
        history.append({"epoch":epoch,"train_loss":tr[0],"train_dice":tr[1],"train_iou":tr[2],
                        "val_loss":va[0],"val_dice":va[1],"val_iou":va[2]})
        print(f"{name} {epoch}/{epochs} | train loss {tr[0]:.4f} Dice {tr[1]:.4f} IoU {tr[2]:.4f} "
              f"| val loss {va[0]:.4f} Dice {va[1]:.4f} IoU {va[2]:.4f}")
        if va[1]>best:
            best=va[1]
            torch.save({"model_name":name,"model_state_dict":model.state_dict(),
                        "image_size":SEGMENTATION_SIZE,"best_val_dice_pseudomask":best,"epoch":epoch},path)
    print("Saved:",path,"Best validation Dice vs pseudo-mask:",round(best,4))
    return model,pd.DataFrame(history),path

# Train U-Net:
SEG_MODEL_NAME="UNet"
seg_model, seg_history, seg_checkpoint = train_segmentation(SEG_MODEL_NAME)


## 12. Plot segmentation training curves

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(seg_history.train_loss,label="train"); ax[0].plot(seg_history.val_loss,label="validation")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(seg_history.train_dice,label="train"); ax[1].plot(seg_history.val_dice,label="validation")
ax[1].set_title("Dice against pseudo-masks"); ax[1].legend()
plt.show()

# To train DeepLabV3+ as well, run:
# SEG_MODEL_NAME="DeepLabV3Plus"
# deeplab_model, deeplab_history, deeplab_checkpoint = train_segmentation(SEG_MODEL_NAME)


## 13. Optional expert-mask test evaluation

In [ ]:
@torch.no_grad()
def evaluate_expert_test(checkpoint_path, test_images_dir, expert_masks_dir):
    ds=SegmentationDataset(test_images_dir,expert_masks_dir,size=SEGMENTATION_SIZE,augment=False)
    loader=DataLoader(ds,batch_size=1,shuffle=False)
    ckpt=torch.load(checkpoint_path,map_location=DEVICE)
    model=build_segmentation_model(ckpt["model_name"])
    model.load_state_dict(ckpt["model_state_dict"]); model.eval()
    rows=[]
    for x,y,paths in loader:
        logits=model(x.to(DEVICE)); pred=(torch.sigmoid(logits)>=.5).bool(); truth=y.to(DEVICE).bool()
        tp=(pred&truth).sum().item(); fp=(pred&~truth).sum().item(); fn=(~pred&truth).sum().item()
        rows.append({"image":Path(paths[0]).name,
                     "Dice":2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 1.0,
                     "IoU":tp/(tp+fp+fn) if tp+fp+fn else 1.0,
                     "Precision":tp/(tp+fp) if tp+fp else 1.0,
                     "Recall":tp/(tp+fn) if tp+fn else 1.0})
    df=pd.DataFrame(rows)
    print("Macro-average expert test metrics:\n",df[["Dice","IoU","Precision","Recall"]].mean().round(4))
    out=os.path.join(OUTPUT_DIR,ckpt["model_name"]+"_expert_test_metrics.csv")
    df.to_csv(out,index=False); print("Saved per-image metrics:",out)
    return df

# Uncomment only when the paths point to a held-out expert-annotated test set:
# expert_metrics = evaluate_expert_test(seg_checkpoint, TEST_IMAGES_DIR, EXPERT_MASKS_DIR)


## 14. Visualize segmentation predictions

In [ ]:
@torch.no_grad()
def visualize_segmentation(checkpoint_path, image_dir, mask_dir, n=4):
    ds=SegmentationDataset(image_dir,mask_dir,size=SEGMENTATION_SIZE,augment=False)
    ckpt=torch.load(checkpoint_path,map_location=DEVICE)
    model=build_segmentation_model(ckpt["model_name"])
    model.load_state_dict(ckpt["model_state_dict"]); model.eval()
    chosen=random.sample(range(len(ds)),min(n,len(ds)))
    fig,axes=plt.subplots(len(chosen),3,figsize=(10,3.5*len(chosen)))
    if len(chosen)==1: axes=np.expand_dims(axes,0)
    for r,i in enumerate(chosen):
        x,y,path=ds[i]; pred=(torch.sigmoid(model(x.unsqueeze(0).to(DEVICE)))[0,0].cpu().numpy()>=.5)
        axes[r,0].imshow(x[0].numpy(),cmap="gray"); axes[r,0].set_title(Path(path).name)
        axes[r,1].imshow(y[0].numpy(),cmap="gray"); axes[r,1].set_title("Target mask")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("Predicted mask")
        for a in axes[r]: a.axis("off")
    plt.tight_layout(); plt.show()

visualize_segmentation(seg_checkpoint, TRAIN_IMAGES_DIR, PSEUDO_MASKS_DIR, n=4)


## 15. Final checklist
- Classification folder names must match the intended labels; inspect `train_full.class_to_idx`.
- Set `TRAIN_IMAGES_DIR` to **training images only** before generating pseudo-masks.
- Do not create pseudo-masks from the expert test set.
- For scientific reporting, use patient-level splits and report expert-mask test metrics.
- Grad-CAM is coarse localization, not ground-truth segmentation. Thresholded CAM masks may be noisy.
- The notebook's simple preprocessing is a baseline; check image orientation, contrast, normalization, and mask alignment for your dataset.
